In [ ]:
import os
import openai
import constantes as c
import callingLLMs as llm
import annotation as ann

In [2]:
# Chemins
path_to_api_key = "/Users/charles-/Downloads/clefs/OPENROUTER_KEY_PERSO.txt"
path_to_res_folder = "./results_v2.2"

## Calling LLMs

In [3]:
with open(path_to_api_key, "r") as file:
    API_KEY = file.read().strip()
client = llm.get_client(API_KEY)

In [4]:
os.makedirs(path_to_res_folder, exist_ok=True)
test = llm.run_all(prompt=c.PROMPT2, models_by_category={"test": ["google/gemma-4-31b-it"]},
                   n_calls=1, client=client, save_path=f"{path_to_res_folder}/test.jsonl")
print(test["answer"][0])

0 appel(s) déjà faits, 1 à effectuer (sauvegarde : ./results_v2.2/test.jsonl)
R : Tout occupant d'un espace public doit être en mesure de justifier son identité sur demande d'un agent assermenté, sous peine d'expulsion immédiate du périmètre.

N : On s'attendrait à ce que la mort d'une personne mette fin à son statut d'occupant et rende son identité hors de propos pour la gestion de l'espace.

Énoncé : Le policier demande la carte d'identité au corps gisant sur le trottoir. Le médecin légiste tend le document trouvé dans la poche du défunt. L'agent vérifie l'adresse, constate que le domicile se situe hors de la commune et ordonne aux pompiers de transporter le cadavre au-delà de la limite municipale.


In [5]:
os.makedirs(path_to_res_folder, exist_ok=True)
df = llm.run_all(prompt=c.PROMPT2, models_by_category=c.MODELES, client=client, n_calls=3,
                 save_path=f"{path_to_res_folder}/generations.jsonl", max_workers=4)
df.to_csv(f"{path_to_res_folder}/generations.csv", index=False)

0 appel(s) déjà faits, 75 à effectuer (sauvegarde : ./results_v2.2/generations.jsonl)


In [11]:
erreurs = df[df["answer"].str.startswith("ERROR")]
print(f"{len(df)} réponses, dont {len(erreurs)} erreurs")
print(erreurs.groupby("model")["answer"].first().str[:150].to_string())

75 réponses, dont 2 erreurs
model
mistralai/mistral-large-2512    ERROR: RateLimitError: Error code: 429 - Provi...


In [12]:
print(erreurs.groupby("model")["answer"].first().to_string())

ids = sorted(m.id for m in client.models.list())
for motif in ["deepseek-r1", "lfm", "qwq"]:
    print(motif, "→", [i for i in ids if motif in i])
    

model
mistralai/mistral-large-2512    ERROR: RateLimitError: Error code: 429 - Provi...
deepseek-r1 → ['deepseek/deepseek-r1-0528']
lfm → ['liquid/lfm-2.5-24b-a2b', 'liquid/lfm-3-1.7b', 'liquid/lfm-3-1.7b:free']
qwq → []


In [13]:
for m in client.models.list():
    if "lfm" in m.id:
        extra = m.model_extra or {}
        raisonne = "reasoning" in (extra.get("supported_parameters") or [])
        print(m.id, "| raisonne :", raisonne, "|", (extra.get("description") or "")[:150])

liquid/lfm-3-1.7b | raisonne : True | A compact 1.7B-parameter text model with a 256K-token context, post-trained for agents, tool calling, and structured outputs. It reasons before it ans
liquid/lfm-3-1.7b:free | raisonne : True | A compact 1.7B-parameter text model with a 256K-token context, post-trained for agents, tool calling, and structured outputs. It reasons before it ans
liquid/lfm-2.5-24b-a2b | raisonne : True | LFM2.5-24B-A2B is a mixture-of-experts reasoning model from Liquid AI with 24B total and 2.45B active parameters. It always reasons before answering a


## Synthesizing into table

In [16]:
# Un fichier Excel par annotateur : ordre mélangé, modèles masqués, critères en 0/1
ann.exporter_annotation(df, annotateurs=("Casanove", "Charles"), dossier=path_to_res_folder)

[PosixPath('results_v2.2/annotation_Casanove.xlsx'),
 PosixPath('results_v2.2/annotation_Charles.xlsx')]

In [17]:
# Après annotation (fichiers enregistrés dans Excel ou LibreOffice) : accord critère par critère
ann.comparer_annotations(f"{path_to_res_folder}/annotation_Casanove.xlsx",
                         f"{path_to_res_folder}/annotation_Charles.xlsx")

,critère,kappa,n
0,1a Cadre nommable,None,0
1,1b Non résolu,None,0
2,2a Règle formulable,None,0
3,2b Monde consistant,None,0
4,3a Rigueur valide,None,0
5,3b Sans intérêt local,None,0
6,3c Productif,None,0
